# Hypothesis Testing

## Objective

The EDA identified several categorical and numerical variables that show
differences in fraud prevalence or distribution between fraudulent and
legitimate transactions.

Hypothesis testing will be used to determine whether selected observed
relationships have statistical evidence supporting them.

The tests will focus on:

- Categorical variables associated with fraud
- Transaction amount differences
- Temporal variation in fraud prevalence

Statistical significance will be evaluated alongside the magnitude and
practical relevance of the observed differences.

### Tests

1. Chi-square test — `id_35` vs `isFraud`
2. Chi-square test — `card6` vs `isFraud`
3. Chi-square test — `ProductCD` vs `isFraud`
4. Mann–Whitney U test — `TransactionAmt` vs `isFraud`
5. Chi-square test — `time_week` vs `isFraud`

In [15]:
import pandas as pd
df = pd.read_csv("../data/processed/analytical_dataset.csv")
print("Analytical dataset shape:", df.shape)

Analytical dataset shape: (590540, 437)


## Hypothesis Testing Plan

The EDA identified several categorical and numerical variables that show
differences in fraud prevalence or distribution between fraudulent and
legitimate transactions.

Hypothesis testing will be used to determine whether these observed
relationships have statistical evidence supporting them.

The analysis will focus on five selected hypotheses identified from the EDA.

| # | Hypothesis / Research Question | Rationale from EDA | Null Hypothesis (H₀) | Alternative Hypothesis (H₁) | Statistical Test | Why It Matters for the Project | Data-Science Decision |
|---|---|---|---|---|---|---|---|
| 1 | Is `id_35` associated with fraud? | Fraud rates differed across `id_35` categories. | `id_35` and `isFraud` are independent. | `id_35` and `isFraud` are associated. | Chi-square test of independence | Determines whether the observed identity-related fraud pattern has statistical evidence. | Retain as a candidate feature if supported; validate through ML. |
| 2 | Is `card6` associated with fraud? | Credit transactions had a higher observed fraud rate than debit transactions. | `card6` and `isFraud` are independent. | `card6` and `isFraud` are associated. | Chi-square test of independence | Determines whether payment-type differences are statistically supported. | Retain as a candidate categorical feature; avoid manual fraud rules. |
| 3 | Is `ProductCD` associated with fraud? | Fraud prevalence differed substantially across product categories. | `ProductCD` and `isFraud` are independent. | `ProductCD` and `isFraud` are associated. | Chi-square test of independence | Determines whether product-category differences in fraud prevalence are statistically supported. | Retain as a candidate feature; do not assign unsupported meanings to categories. |
| 4 | Do transaction amounts differ between fraudulent and legitimate transactions? | Transaction amounts were right-skewed and showed different distributions between the two groups. | The distributions of `TransactionAmt` are the same for fraudulent and legitimate transactions. | The distributions of `TransactionAmt` differ between the two groups. | Mann–Whitney U test | Determines whether transaction amount distributions differ statistically between fraud and legitimate transactions. | Retain `TransactionAmt`; investigate transformations and non-linear relationships. |
| 5 | Is fraud prevalence associated with transaction period? | Weekly fraud rates varied across the 26-week observation period. | `time_week` and `isFraud` are independent. | `time_week` and `isFraud` are associated. | Chi-square test of independence | Determines whether temporal variation in fraud prevalence has statistical evidence. | Supports time-aware modeling and validation. |

### Significance and Effect Size

A significance level of **α = 0.05** will be used.

For the chi-square tests, **Cramér's V** will be reported as an effect-size
measure. For the Mann–Whitney U test, an appropriate effect-size measure
will also be reported.

Because the dataset contains a large number of observations, statistical
significance alone will not determine whether a relationship is practically
important.

### Interpretation Framework

Each hypothesis will be evaluated using:

**Test statistic → p-value → effect size → statistical conclusion → practical
interpretation → data-science decision**

The hypothesis tests are intended to validate relationships identified during
EDA. A statistically significant relationship does not automatically imply
that a feature will improve machine-learning performance. Final feature
selection will be determined using leakage-safe feature engineering and
chronological model validation.

### Multiple Testing

Because multiple hypotheses are being tested, the results will be interpreted
with consideration of the multiple-testing problem. A significant p-value
will therefore not be treated as sufficient evidence of practical importance
or predictive usefulness by itself.

## Test 1 — Association Between `id_35` and Fraud

### Research Question

Is `id_35` statistically associated with `isFraud`?

### Hypotheses

- **H₀:** `id_35` and `isFraud` are independent.
- **H₁:** `id_35` and `isFraud` are associated.

### Why Chi-square?

Both `id_35` and `isFraud` are categorical variables. Therefore, a
chi-square test of independence is appropriate for testing whether their
observed frequencies differ from what would be expected if the variables
were independent.

In [2]:
# Contingency table: id_35 vs fraud status
id35_table = pd.crosstab(
    df["id_35"],
    df["isFraud"]
)

id35_table

isFraud,0,1
id_35,,
F,55426,7745
T,74337,3477


In [3]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(id35_table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 2888.471367177767
p-value: 0.0
Degrees of freedom: 1


In [4]:
# Calculate Cramér's V effect size
n = id35_table.to_numpy().sum()
phi2 = chi2 / n

cramers_v = (phi2 / min(id35_table.shape[0] - 1, id35_table.shape[1] - 1)) ** 0.5

print("Cramér's V:", cramers_v)

Cramér's V: 0.14313557239737848


### Result & Interpretation

**Observation:**  
The chi-square test produced a χ² statistic of 2888.47 with a p-value < 0.05. Cramér's V was 0.1431.

**Interpretation:**  
We reject the null hypothesis of independence and conclude that `id_35` is statistically associated with `isFraud`. However, the Cramér's V indicates that the strength of this association is relatively weak.

**Data-Science Implication:**  
`id_35` may contain useful information for distinguishing fraudulent transactions from legitimate transactions. However, statistical significance does not by itself establish predictive usefulness.

**Decision:**  
Retain `id_35` as a candidate feature and evaluate its contribution during leakage-safe temporal model validation.

## Test 2 — Association Between `card6` and Fraud

### Research Question

Is `card6` statistically associated with `isFraud`?

### Hypotheses

- **H₀:** `card6` and `isFraud` are independent.
- **H₁:** `card6` and `isFraud` are associated.

### Why Chi-square?

Both `card6` and `isFraud` are categorical variables. Therefore, a
chi-square test of independence is appropriate for testing whether the
distribution of fraud status differs across the categories of `card6`.

### Why It Matters for the Project

`card6` represents the type of payment card and may contain useful
information for distinguishing fraudulent from legitimate transactions.

However, statistical association does not imply causation or guarantee
predictive usefulness. Its final value as a model feature will be assessed
during leakage-safe temporal model validation.

In [5]:
# Contingency table: card6 vs fraud status
card6_table = pd.crosstab(
    df["card6"],
    df["isFraud"]
)

card6_table

isFraud,0,1
card6,,
charge card,15,0
credit,139036,9950
debit,429264,10674
debit or credit,30,0


In [7]:
from scipy.stats import chi2_contingency

# Perform chi-square test of independence
chi2, p_value, dof, expected = chi2_contingency(card6_table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 5957.032292414726
p-value: 0.0
Degrees of freedom: 3


In [8]:
# Calculate Cramér's V effect size
n = card6_table.to_numpy().sum()
phi2 = chi2 / n

cramers_v = (
    phi2 / min(card6_table.shape[0] - 1, card6_table.shape[1] - 1)
) ** 0.5

print("Cramér's V:", cramers_v)

Cramér's V: 0.10057007150101338


### Test 2 — Conclusion

**Observation:**  
The chi-square test indicates a statistically significant association between
`card6` and `isFraud` (p < 0.05). Cramér's V is approximately 0.101, indicating
a small association.

**Interpretation:**  
The distribution of fraud differs across `card6` categories, but the strength
of the relationship is relatively small.

**Data-Science Implication:**  
`card6` may provide useful information to a fraud detection model, but it is
unlikely to be a strong standalone predictor of fraud.

**Decision:**  
Retain `card6` as a candidate feature. Its final predictive usefulness will be
evaluated during leakage-safe temporal model validation.

## Test 3 — Association Between `ProductCD` and Fraud

### Research Question

Is `ProductCD` statistically associated with `isFraud`?

### Hypotheses

- **H₀:** `ProductCD` and `isFraud` are independent.
- **H₁:** `ProductCD` and `isFraud` are associated.

### Why Chi-square?

Both `ProductCD` and `isFraud` are categorical variables. Therefore, a
chi-square test of independence is appropriate for testing whether the
distribution of fraud status differs across the product categories.

### Why It Matters for the Project

EDA showed substantial differences in fraud rates across `ProductCD`
categories. This test determines whether those differences are statistically
significant.

However, statistical association does not imply causation or guarantee
predictive usefulness. Its final value as a model feature will be assessed
during leakage-safe temporal model validation.

In [9]:
# Step 1 — Contingency table
productcd_table = pd.crosstab(
    df["ProductCD"],
    df["isFraud"]
)

productcd_table

isFraud,0,1
ProductCD,,
C,60511,8008
H,31450,1574
R,36273,1426
S,10942,686
W,430701,8969


In [10]:
from scipy.stats import chi2_contingency

# Perform chi-square test of independence
chi2, p_value, dof, expected = chi2_contingency(productcd_table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)


Chi-square statistic: 16742.17152945829
p-value: 0.0
Degrees of freedom: 4


In [11]:
# Calculate Cramér's V effect size
n = productcd_table.to_numpy().sum()
phi2 = chi2 / n

cramers_v = (
    phi2 / min(productcd_table.shape[0] - 1, productcd_table.shape[1] - 1)
) ** 0.5

print("Cramér's V:", cramers_v)

Cramér's V: 0.16837640539825974


### Test 3 — Conclusion

**Observation:**  
The chi-square test indicates a statistically significant association between
`ProductCD` and `isFraud` (p < 0.05). Cramér's V is approximately 0.168,
indicating a small-to-moderate association.

**Interpretation:**  
Fraud rates differ substantially across `ProductCD` categories, with some
categories showing considerably higher fraud rates than others.

**Data-Science Implication:**  
`ProductCD` contains potentially useful information for distinguishing
fraudulent from legitimate transactions and may provide more predictive
information than `card6`.

**Decision:**  
Retain `ProductCD` as a candidate feature. Its final predictive usefulness
will be evaluated during leakage-safe temporal model validation.

## Test 4 — Difference in Transaction Amount Between Fraud and Legitimate Transactions

### Research Question

Do fraudulent and legitimate transactions have statistically different
`TransactionAmt` distributions?

### Hypotheses

- **H₀:** The distribution of `TransactionAmt` is the same for fraudulent and
  legitimate transactions.
- **H₁:** The distribution of `TransactionAmt` differs between fraudulent and
  legitimate transactions.

### Why Mann–Whitney U?

`TransactionAmt` is a continuous numerical variable with a strongly skewed
distribution and substantial outliers. The Mann–Whitney U test is a
non-parametric test that compares the distributions of two independent groups
without requiring a normality assumption.

### Why It Matters for the Project

Transaction amount may contain useful information for identifying
fraudulent transactions. However, statistical significance alone does not
determine whether the feature will improve a fraud detection model.

Its final predictive usefulness will be assessed during leakage-safe temporal
model validation.

In [12]:
from scipy.stats import mannwhitneyu

# Separate transaction amounts by fraud status
legitimate_amounts = df.loc[
    df["isFraud"] == 0, "TransactionAmt"
].dropna()

fraud_amounts = df.loc[
    df["isFraud"] == 1, "TransactionAmt"
].dropna()

# Perform Mann–Whitney U test
u_statistic, p_value = mannwhitneyu(
    fraud_amounts,
    legitimate_amounts,
    alternative="two-sided"
)

print("Mann–Whitney U statistic:", u_statistic)
print("p-value:", p_value)

Mann–Whitney U statistic: 5858540820.5
p-value: 0.22590783497976064


In [13]:
# Calculate rank-biserial correlation as the effect size
n_fraud = len(fraud_amounts)
n_legitimate = len(legitimate_amounts)

rank_biserial = (
    2 * u_statistic / (n_fraud * n_legitimate)
) - 1

print("Rank-biserial correlation:", rank_biserial)

Rank-biserial correlation: -0.004949892671515577


### Test 4 — Conclusion

**Observation:**  
The Mann–Whitney U test produced a p-value of approximately 0.226, which is
greater than the significance level of 0.05. The rank-biserial correlation
is approximately -0.005, indicating an effect size close to zero.

**Interpretation:**  
There is insufficient statistical evidence to conclude that the overall
distribution of `TransactionAmt` differs between fraudulent and legitimate
transactions.

**Data-Science Implication:**  
Although transaction amount showed some differences in descriptive EDA,
those differences are not statistically significant at the overall
distribution level. This suggests that transaction amount alone may provide
limited discriminatory information.

**Decision:**  
Retain `TransactionAmt` as a candidate feature because tree-based models may
still exploit nonlinear relationships or interactions. However, do not
consider it a strong standalone fraud indicator based on this hypothesis test.
Its final predictive contribution will be assessed during temporal model
validation.

## Test 5 — Association Between Time Period and Fraud

### Research Question

Is `time_week` statistically associated with `isFraud`?

### Hypotheses

- **H₀:** `time_week` and `isFraud` are independent.
- **H₁:** `time_week` and `isFraud` are associated.

### Why Chi-square?

Both `time_week` and `isFraud` are categorical variables. Therefore, a
chi-square test of independence is appropriate for determining whether the
distribution of fraud status differs across weekly time periods.

### Why It Matters for the Project

EDA showed that fraud rates varied across the 26-week period. A statistically
significant association would indicate that fraud prevalence is not constant
across time.

This does not imply that time itself causes fraud. Temporal patterns may
reflect changes in transaction behavior, fraud strategies, or other underlying
factors.

Temporal patterns are also important for model validation because a fraud
detection model should be evaluated on future transactions rather than using
a random split alone.

In [17]:
# Create weekly time periods for the analysis.
df["transaction_day"] = (
    df["TransactionDT"] - df["TransactionDT"].min()
) / (24 * 60 * 60)
df["time_week"] = (df["transaction_day"] // 7).astype(int)

# Step 1 — Contingency table
time_week_table = pd.crosstab(
    df["time_week"],
    df["isFraud"]
)

time_week_table

isFraud,0,1
time_week,,
0,26792,804
1,27746,717
2,34832,869
3,34182,724
4,23650,889
5,20116,803
6,19741,823
7,18899,862
8,20505,927


In [18]:
from scipy.stats import chi2_contingency

# Perform chi-square test of independence
chi2, p_value, dof, expected = chi2_contingency(time_week_table)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)
print("Degrees of freedom:", dof)

Chi-square statistic: 918.1638069871987
p-value: 1.2875635099548486e-177
Degrees of freedom: 25


In [19]:
# Calculate Cramér's V effect size
n = time_week_table.to_numpy().sum()
phi2 = chi2 / n

cramers_v = (
    phi2 / min(time_week_table.shape[0] - 1, time_week_table.shape[1] - 1)
) ** 0.5

print("Cramér's V:", cramers_v)

Cramér's V: 0.03943078514471564


### Test 5 — Conclusion

**Observation:**  
The chi-square test indicates a statistically significant association between
`time_week` and `isFraud` (p < 0.05). Cramér's V is approximately 0.039,
indicating a very small association.

**Interpretation:**  
Fraud prevalence varies across the 26 weekly time periods, but the overall
strength of the association is weak.

**Data-Science Implication:**  
Temporal information may still provide useful context for fraud detection,
but `time_week` should not be considered a strong standalone predictor.
Temporal patterns are particularly important for designing realistic
time-aware model validation.

**Decision:**  
Retain temporal information as a candidate feature and use time-aware
validation. The final predictive contribution of temporal features will be
determined through model validation rather than statistical significance
alone.

## Multiple Testing Consideration

Five statistical hypothesis tests were conducted as part of the exploratory
analysis.

Because multiple hypotheses were tested, the probability of obtaining at least
one statistically significant result by chance increases. Therefore, the
reported p-values should be interpreted in the context of multiple testing.

The hypothesis tests are used as exploratory evidence rather than as the sole
basis for feature selection. Effect sizes and domain relevance are considered
alongside statistical significance.

Final feature selection will be based on leakage-safe temporal model
validation and predictive performance rather than p-values alone.

## Final Hypothesis Testing Results

The following five hypothesis tests were conducted to investigate statistical
relationships between selected transaction characteristics and fraud status.

| Test | Variable | Statistical Test | p-value | Association | Effect Size | Decision |
|------|----------|------------------|---------|-------------|-------------|----------|
| 1 | `id_35` | Chi-square | < 0.05 | Small-to-moderate | Cramér's V = 0.1431 | Reject H₀ |
| 2 | `card6` | Chi-square | < 0.05 | Small | Cramér's V = 0.101 | Reject H₀ |
| 3 | `ProductCD` | Chi-square | < 0.05 | Small-to-moderate | Cramér's V = 0.168 | Reject H₀ |
| 4 | `TransactionAmt` | Mann–Whitney U | 0.226 | Negligible | Rank-biserial = -0.005 | Fail to reject H₀ |
| 5 | `time_week` | Chi-square | < 0.05 | Very small | Cramér's V = 0.039 | Reject H₀ |

### Significance Level

All hypothesis tests were evaluated using a significance level of α = 0.05.

A statistically significant result indicates evidence against the null
hypothesis, but statistical significance alone does not establish practical
importance or predictive usefulness.

## Data-Science Conclusions

### 1. Categorical features showed different levels of association with fraud

`ProductCD`, `card6`, and `id_35` showed statistically significant associations
with `isFraud`. However, the effect sizes varied across features.

`ProductCD` showed the strongest association among the tested categorical
variables, while `card6` showed a smaller association.

This demonstrates that statistical significance and association strength should
be considered separately.

### 2. Transaction amount was not statistically significant in the overall distribution test

The Mann–Whitney U test did not provide sufficient evidence that the overall
`TransactionAmt` distributions differ between fraudulent and legitimate
transactions.

Therefore, transaction amount should not be described as a strong standalone
fraud indicator based on this test.

It will nevertheless be retained as a candidate feature because machine
learning models can capture nonlinear relationships and interactions that are
not represented by a single overall distribution comparison.

### 3. Fraud prevalence varied across time

`time_week` showed a statistically significant association with fraud status,
but its Cramér's V indicated a very small association.

This suggests that temporal variation exists, but its overall strength is
limited.

Temporal information remains important for model development because fraud
patterns can change over time and because realistic fraud detection systems
must be evaluated on future transactions.

### 4. Statistical testing will not determine final feature selection

These hypothesis tests provide exploratory statistical evidence.

Final feature selection will additionally consider:

- Data leakage
- Missingness and data quality
- Feature redundancy
- Temporal stability
- Predictive performance
- Model interpretability
- Business usefulness

Therefore, a statistically significant feature will not automatically be
included in the final model, and a statistically non-significant feature will
not automatically be removed.

### Overall Decision

The hypothesis-testing analysis supports retaining the selected features as
candidate predictors for the next stage of the pipeline.

The project will now move from statistical analysis to feature engineering,
where features will be transformed and evaluated using leakage-safe temporal
validation.
